# Pandalyzer — Profiler Test Notebook
## Objective
Test Pandalyzer's read-only profiler on synthetic data and the messy retail CSV when available.

## Workflow
1. Load a dataset.
2. Inspect structure and missingness.
3. Run the profiler.
4. Verify profile metrics against direct pandas calculations.
5. Verify profiling does not mutate the input.

## 1. Imports and dataset setup

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd

_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (p for p in _candidates if (p / "database").is_dir() and (p / "src" / "autoclean").is_dir()),
    Path.cwd(),
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from autoclean.cleaner import remove_duplicate_rows, handle_missing_values, clean_dataframe
from autoclean.profiler import profile_dataframe, compare_data_quality
from autoclean.validators import validate_missing_values, validate_required_columns, validate_column_types, validate_numeric_ranges
from autoclean.connectors import load_csv, save_csv, save_quality_report

csv_path = PROJECT_ROOT / "database" / "messy_retail_orders.csv"
print("Project root:", PROJECT_ROOT)
print("Dataset available:", csv_path.exists())

if csv_path.exists():
    df = load_csv(csv_path)
else:
    df = pd.DataFrame({"id":[1,2,2,3], "amount":[10.0,None,None,40.0], "category":["a","b","b","d"]})
print("Dataset shape:", df.shape)

Project root: /run/media/amateur/MATRIX/Pandalyzer/AutoClean
Dataset available: True
Dataset shape: (10000, 20)


## 2. Inspect dataset structure

In [2]:
print("Columns:", df.columns.tolist())
print("Dtypes:")
display(df.dtypes.rename("data_type").to_frame())
print("Preview:")
display(df.head())

Columns: ['order_id', 'order_date', 'customer_id', 'product_id', 'category', 'quantity', 'unit_price', 'discount_pct', 'order_amount', 'payment_method', 'order_status', 'city', 'state', 'sales_channel', 'customer_email', 'delivery_days', 'rating', 'coupon_code', 'is_returned', 'notes']
Dtypes:


,data_type
order_id,str
order_date,str
customer_id,str
product_id,str
category,str
quantity,str
unit_price,str
discount_pct,str
order_amount,str
payment_method,str


Preview:


,order_id,order_date,customer_id,product_id,category,quantity,unit_price,discount_pct,order_amount,payment_method,order_status,city,state,sales_channel,customer_email,delivery_days,rating,coupon_code,is_returned,notes
0,ORD009826,2024-06-19 16:56:11,CUST01648,NaN,Grocery,5.0,96.55,30.0,NaN,Credit Card,Cancelled,Mumbai,Maharashtra,Website,not-an-email,12.0,1.0,NaN,No,NaN
1,ORD004017,2025-03-18 10:42:56,CUST01249,\tPRD0244\t,Home & Kitchen,NaN,1567.44,5.0,4467.2,COD,\tShipped\t,Hyderabad,Telangana,Website,customer1175@example.com,NaN,3.0,NaN,No,NaN
2,ORD008732,2026-09-30 12:23:18,CUST00211,PRD0160,Clothing,2.0,958.08,0.0,1916.16,Credit Card,Returned,lucknow,Uttar Pradesh,Store,NaN,11.0,1.0,NaN,No,Gift order
3,ORD009698,2024-01-18 19:02:44,CUST00406,PRD0178,Electronics,2.0,421.03,0.0,NaN,Wallet,Returned,Bengaluru,NaN,NaN,@example.com,9.0,2.0,FESTIVE25,NaN,Customer requested update
4,ORD004226,NaN,CUST00610,PRD0235,Sports,2.0,1653.18,20.0,2645.09,Cash,Cancelled,Delhi,Delhi,Website,\tcustomer780@example.com \t,8.0,5.0,NaN,NaN,Gift order


## 3. Run the profiler

In [3]:
df_before_profiling = df.copy(deep=True)
report = profile_dataframe(df)
print("Rows:", report["row_count"])
print("Columns:", report["column_count"])
print("Exact duplicate rows:", report["duplicate_row_count"])
display(report["columns"])

Rows: 10000
Columns: 20
Exact duplicate rows: 0


,data_type,missing_count,missing_percent,distinct_count
order_id,str,96,0.96,9595
order_date,str,430,4.30,9329
customer_id,str,975,9.75,2747
product_id,str,832,8.32,1008
category,str,1021,10.21,242
quantity,str,166,1.66,15
unit_price,str,841,8.41,8173
discount_pct,str,917,9.17,202
order_amount,str,848,8.48,8561
payment_method,str,297,2.97,30


## 4. Verify profiler counts against pandas

In [4]:
assert report["row_count"] == len(df)
assert report["column_count"] == len(df.columns)
assert report["duplicate_row_count"] == int(df.duplicated().sum())
assert report["columns"]["missing_count"].equals(df.isna().sum())
assert report["columns"]["distinct_count"].equals(df.nunique(dropna=True))
print("PASS: profiler metrics match pandas")

PASS: profiler metrics match pandas


## 5. Sort columns by missingness

In [5]:
missing_summary = report["columns"].sort_values("missing_percent", ascending=False)
display(missing_summary)

,data_type,missing_count,missing_percent,distinct_count
coupon_code,str,4821,48.21,10
notes,str,4581,45.81,125
customer_email,str,1134,11.34,3188
delivery_days,str,1109,11.09,88
category,str,1021,10.21,242
order_status,str,993,9.93,226
customer_id,str,975,9.75,2747
rating,str,966,9.66,18
state,str,933,9.33,268
sales_channel,str,931,9.31,187


## 6. Show columns with missing values

In [6]:
columns_with_missing = missing_summary[missing_summary["missing_count"] > 0]
display(columns_with_missing)
print("Columns with missing values:", len(columns_with_missing))

,data_type,missing_count,missing_percent,distinct_count
coupon_code,str,4821,48.21,10
notes,str,4581,45.81,125
customer_email,str,1134,11.34,3188
delivery_days,str,1109,11.09,88
category,str,1021,10.21,242
order_status,str,993,9.93,226
customer_id,str,975,9.75,2747
rating,str,966,9.66,18
state,str,933,9.33,268
sales_channel,str,931,9.31,187


Columns with missing values: 20


## 7. Calculate overall missingness

In [7]:
total_cells = df.shape[0] * df.shape[1]
missing_cells = int(df.isna().sum().sum())
overall_missing_percent = missing_cells / total_cells * 100 if total_cells else 0.0
print("Missing cells:", missing_cells)
print(f"Overall missingness: {overall_missing_percent:.2f}%")

Missing cells: 23018
Overall missingness: 11.51%


## 8. Inspect duplicate rows

In [8]:
duplicate_mask = df.duplicated(keep=False)
print("Rows belonging to duplicate groups:", int(duplicate_mask.sum()))
display(df.loc[duplicate_mask].head(20))

Rows belonging to duplicate groups: 0


,order_id,order_date,customer_id,product_id,category,quantity,unit_price,discount_pct,order_amount,payment_method,order_status,city,state,sales_channel,customer_email,delivery_days,rating,coupon_code,is_returned,notes


## 9. Verify profiling is read-only

In [9]:
profile_dataframe(df)
pd.testing.assert_frame_equal(df, df_before_profiling)
print("PASS: profiling did not modify the DataFrame")

PASS: profiling did not modify the DataFrame


## 10. Final summary

In [10]:
print("=" * 45)
print("PANDALYZER PROFILING SUMMARY")
print("=" * 45)
print("Dataset:", csv_path.name if csv_path.exists() else "synthetic fallback")
print(f"Rows: {report['row_count']:,}")
print(f"Columns: {report['column_count']:,}")
print(f"Missing cells: {missing_cells:,}")
print(f"Overall missingness: {overall_missing_percent:.2f}%")
print(f"Duplicate rows: {report['duplicate_row_count']:,}")
print("PASS: profiler notebook completed")

PANDALYZER PROFILING SUMMARY
Dataset: messy_retail_orders.csv
Rows: 10,000
Columns: 20
Missing cells: 23,018
Overall missingness: 11.51%
Duplicate rows: 0
PASS: profiler notebook completed
